In [ ]:
import pandas as pd

df = pd.read_excel(r"/kaggle/working/datasets-dgtx/DATA_DG.xlsx")   # hoặc path đúng của bạn

assert {"content", "summary", "grade"}.issubset(df.columns)
df = df.dropna().reset_index(drop=True)


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "VietAI/vit5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer.model_max_length)  # ~512 tokens

from datasets import Dataset

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 256

def preprocess(batch):
    prompts = [
        f"Tóm tắt văn bản cho học sinh lớp {g}: {c}"
        for c, g in zip(batch["content"], batch["grade"])
    ]

    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LEN,
    )

    labels = tokenizer(
        batch["summary"],
        truncation=True,
        max_length=MAX_TARGET_LEN,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

tokenized_ds = dataset.map(
    preprocess,
    batched=True,   # ⭐ BẮT BUỘC
    remove_columns=dataset["train"].column_names
)

print(tokenized_ds["train"].column_names)

In [ ]:
# ============================================
# CELL 1: OPTUNA HYPERPARAMETER TUNING
# ============================================

import optuna
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = 1
    epochs = trial.suggest_int("num_train_epochs", 3, 8)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    torch.cuda.empty_cache()
    import gc
    gc.collect()

    args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        eval_strategy="no",
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="no",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
        fp16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        ddp_find_unused_parameters=False,
        report_to="none"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=None
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=None,
    )

    trainer.train()
    train_loss = trainer.state.log_history[-1].get("train_loss", float('inf'))
    
    del model
    del trainer
    del data_collator
    del args
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    return train_loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

best_params = study.best_params
print("=" * 50)
print("BEST HYPERPARAMETERS:")
print("=" * 50)
for key, value in best_params.items():
    print(f"{key}: {value}")
print("=" * 50)

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    import numpy as np
    
    # preds từ Trainer là logits, cần lấy argmax để có token IDs
    if isinstance(preds, tuple):
        preds = preds[0]
    
    if isinstance(preds, np.ndarray):
        # Lấy token ID có xác suất cao nhất (argmax)
        if preds.ndim == 3:  # (batch_size, seq_len, vocab_size)
            preds = np.argmax(preds, axis=-1)
        elif preds.ndim > 3:
            preds = preds.reshape(preds.shape[0], -1)
            preds = np.argmax(preds, axis=-1)
    
    # Xử lý labels - loại bỏ padding tokens (-100)
    if isinstance(labels, np.ndarray):
        labels = labels.tolist()
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Decode labels - loại bỏ -100 trước khi decode
    decoded_labels = []
    for label in labels:
        if isinstance(label, np.ndarray):
            label = label.tolist()
        # Loại bỏ -100 (ignore index)
        valid_label = [l for l in label if l != -100]
        decoded_labels.append(tokenizer.decode(valid_label, skip_special_tokens=True))

    # Đảm bảo decoded_preds và decoded_labels là list các string
    decoded_preds = [str(p) if p else "" for p in decoded_preds]
    decoded_labels = [str(l) if l else "" for l in decoded_labels]

    # -------- ROUGE --------
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # -------- BERTScore --------
    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="vi",
        model_type="bert-base-multilingual-cased"
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_precision": sum(bert_result["precision"]) / len(bert_result["precision"]),
        "bertscore_recall": sum(bert_result["recall"]) / len(bert_result["recall"]),
        "bertscore_f1": sum(bert_result["f1"]) / len(bert_result["f1"]),
    }
    
import pandas as pd

def save_metrics(trainer, filename="metrics.csv"):
    logs = trainer.state.log_history
    df_metrics = pd.DataFrame(logs)
    df_metrics.to_csv(filename, index=False)
    return df_metrics

In [ ]:
# ============================================
# CELL 2: FINAL TRAINING VỚI BEST PARAMETERS
# ============================================

from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
import torch
import os

torch.cuda.empty_cache()

# Kiểm tra xem có checkpoint để resume không
resume_from_checkpoint = None
checkpoint_dir = "./vit5_final"
if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint-")]
    if checkpoints:
        # Lấy checkpoint mới nhất
        checkpoint_numbers = [int(c.split("-")[1]) for c in checkpoints]
        latest_checkpoint = f"checkpoint-{max(checkpoint_numbers)}"
        resume_from_checkpoint = os.path.join(checkpoint_dir, latest_checkpoint)
        print(f"Tìm thấy checkpoint: {resume_from_checkpoint}")
        print("Sẽ tiếp tục training từ checkpoint này...")
    else:
        print("Không tìm thấy checkpoint, bắt đầu training từ đầu")
else:
    print("Không tìm thấy thư mục checkpoint, bắt đầu training từ đầu")

best_learning_rate = 0.0003290826760268271
best_num_train_epochs = 5
best_weight_decay = 0.004878586459798251

final_args = TrainingArguments(
    output_dir="./vit5_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=best_learning_rate,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=best_num_train_epochs,
    weight_decay=best_weight_decay,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
    save_total_limit=5,
    report_to="none"
)

# Load model từ checkpoint nếu có, nếu không thì từ pretrained
if resume_from_checkpoint:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(resume_from_checkpoint)
else:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

final_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=final_model,
    padding=True,
    pad_to_multiple_of=None
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=final_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("=" * 50)
print("BẮT ĐẦU TRAINING VỚI BEST PARAMETERS")
if resume_from_checkpoint:
    print(f"Resume từ checkpoint: {resume_from_checkpoint}")
print("=" * 50)

if resume_from_checkpoint:
    final_trainer.train(resume_from_checkpoint=resume_from_checkpoint)
else:
    final_trainer.train()

final_trainer.save_model("./vit5_grade_summary")
tokenizer.save_pretrained("./vit5_grade_summary")

print("=" * 50)
print("TRAINING HOÀN TẤT!")
print("Model đã được lưu tại: ./vit5_grade_summary")
print("=" * 50)

metrics_df = save_metrics(final_trainer, "vit5_metrics.csv")
print("\nMetrics:")
print(metrics_df.head())
print("\nColumns:", metrics_df.columns.tolist())

import matplotlib.pyplot as plt

eval_df = metrics_df[metrics_df["eval_rouge1"].notna()].copy()
if "epoch" not in eval_df.columns or eval_df["epoch"].isna().all():
    if "step" in eval_df.columns:
        steps_per_epoch = len(tokenized_ds["train"]) // (final_args.per_device_train_batch_size * final_args.gradient_accumulation_steps)
        eval_df["epoch"] = eval_df["step"] / steps_per_epoch
    else:
        eval_df["epoch"] = range(len(eval_df))

if len(eval_df) > 0:
    plt.figure(figsize=(10,6))
    if "eval_rouge1" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rouge1"], label="ROUGE-1")
    if "eval_rougeL" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rougeL"], label="ROUGE-L")
    bertscore_col = "eval_bertscore_f1" if "eval_bertscore_f1" in eval_df.columns else "bertscore_f1"
    if bertscore_col in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df[bertscore_col], label="BERTScore-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics over Epochs")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Warning: No evaluation data found for plotting")

In [1]:
# ============================================
# CELL: TEST MODEL
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

# Đường dẫn đến model đã train
MODEL_PATH = "../Model_DG/vit5_grade_summary"
ORIGINAL_MODEL = "VietAI/vit5-base"

# Kiểm tra xem model có tồn tại không
if not os.path.exists(MODEL_PATH):
    print(f"Model không tồn tại tại: {MODEL_PATH}")
    print("Vui lòng train model trước hoặc kiểm tra đường dẫn")
else:
    print(f"Đang load model từ: {MODEL_PATH}")
    
    # Load tokenizer - thử từ checkpoint trước, nếu lỗi thì dùng model gốc
    # (Tokenizer không thay đổi sau training, chỉ model weights thay đổi)
    try:
        print("Đang load tokenizer từ checkpoint...")
        test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
        print("✓ Tokenizer loaded từ checkpoint (slow tokenizer)")
    except Exception as e1:
        try:
            print(f"⚠ Không thể load slow tokenizer: {str(e1)[:100]}")
            print("Đang thử load fast tokenizer...")
            test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
            print("✓ Tokenizer loaded từ checkpoint (fast tokenizer)")
        except Exception as e2:
            print(f"⚠ Không thể load tokenizer từ checkpoint: {str(e2)[:100]}")
            print(f"⚠ File tokenizer.json có thể bị corrupt")
            print(f"Đang load tokenizer từ model gốc {ORIGINAL_MODEL}...")
            print("(Lưu ý: Tokenizer không thay đổi sau training, nên dùng model gốc vẫn đúng)")
            test_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL)
            print("✓ Tokenizer loaded từ model gốc")
    
    # Load model weights từ checkpoint (đây là phần quan trọng - model đã được fine-tune)
    print("\nĐang load model weights từ checkpoint...")
    test_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
    print("✓ Model weights loaded từ checkpoint")
    
    # Chuyển model sang GPU nếu có
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_model.to(device)
    test_model.eval()
    
    print(f"Model đã được load vào: {device}")
    print("=" * 50)

def calculate_relevance_score(summary_text, original_text):
    """
    Tính điểm độ liên quan giữa tóm tắt và văn bản gốc dựa trên keyword overlap
    
    Returns:
        float: Điểm từ 0.0 đến 1.0 (càng cao càng liên quan)
    """
    import re
    
    stopwords = {'là', 'của', 'và', 'với', 'cho', 'để', 'trong', 'trên', 'từ', 'đến', 'một', 'các', 'những', 'này', 'đó', 'đã', 'được', 'sẽ', 'có', 'không', 'theo', 'sau', 'trước', 'khi', 'nếu', 'thì', 'mà', 'nhưng', 'hoặc'}
    
    def extract_keywords(text):
        text = re.sub(r'[^\w\s]', ' ', text.lower())
        words = text.split()
        keywords = [w for w in words if len(w) > 2 and w not in stopwords]
        return set(keywords)
    
    summary_keywords = extract_keywords(summary_text)
    original_keywords = extract_keywords(original_text)
    
    if len(original_keywords) == 0:
        return 1.0
    
    intersection = len(summary_keywords & original_keywords)
    union = len(summary_keywords | original_keywords)
    
    if union == 0:
        return 0.0
    
    jaccard = intersection / union
    coverage = intersection / len(original_keywords) if len(original_keywords) > 0 else 0.0
    relevance = (jaccard * 0.4 + coverage * 0.6)
    
    return relevance


def summarize(text, grade, max_new_tokens=None, use_sampling=True, temperature=0.9, debug=False):
    """
    Tóm tắt văn bản cho học sinh theo lớp
    
    Args:
        text: Văn bản cần tóm tắt
        grade: Lớp học (1-5)
        max_new_tokens: Số token tối đa (nếu None, tự động tính dựa trên 50% số câu)
        use_sampling: Không dùng nữa (luôn dùng sampling để tạo đa dạng)
        temperature: Độ đa dạng của output (0.1-1.5)
            - 0.7-0.9: Đa dạng vừa phải, cân bằng giữa chất lượng và đa dạng (khuyến nghị)
            - 0.5-0.7: Ít đa dạng hơn, ổn định hơn
            - 0.9-1.2: Đa dạng cao, mỗi lần chạy khác nhau nhiều
            - >1.2: Rất đa dạng nhưng có thể kém chính xác
        debug: Nếu True, in thông tin debug
    
    Returns:
        Tóm tắt văn bản (đã được cắt ở câu hoàn chỉnh và loại bỏ phần không liên quan)
        Mỗi lần chạy sẽ tạo ra tóm tắt khác nhau do sử dụng sampling
    """
    import re
    
    if grade < 1:
        grade = 1
    if grade > 5:
        grade = 5
    
    # Tính số câu trong văn bản gốc
    original_sentences = re.split(r'[.!?]+\s*', text)
    original_sentences = [s.strip() for s in original_sentences if s.strip()]
    num_original_sentences = len(original_sentences)
    
    # Tính max_new_tokens dựa trên 50% số câu
    if max_new_tokens is None:
        avg_tokens_per_sentence = {
            1: 15,
            2: 18,
            3: 22,
            4: 25,
            5: 30  # Tăng lên để model generate dài hơn
        }.get(grade, 25)
        
        target_sentences = max(1, int(num_original_sentences * 0.5))
        max_new_tokens = target_sentences * avg_tokens_per_sentence
        
        # Tăng giới hạn đáng kể để ép model generate dài hơn (vì model được train với MAX_TARGET_LEN=64)
        # Với 21 câu gốc, 50% = ~10-11 câu, mỗi câu 30 tokens = 300-330 tokens
        min_limit = {1: 120, 2: 150, 3: 200, 4: 250, 5: 350}.get(grade, 300)
        max_limit = {1: 300, 2: 400, 3: 500, 4: 650, 5: 800}.get(grade, 700)
        max_new_tokens = max(min_limit, min(max_new_tokens, max_limit))
    
    # Tăng min_new_tokens lên 50% để đảm bảo tóm tắt đủ dài
    min_new_tokens = int(max_new_tokens * 0.5)
    
    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {text}"
    
    inputs = test_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        # Dùng sampling với temperature để tạo sự đa dạng mỗi lần chạy
        # Temperature cao hơn = đa dạng hơn, nhưng có thể kém chính xác hơn
        output = test_model.generate(
            **inputs,
            min_new_tokens=min_new_tokens,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # Bật sampling để tạo đa dạng
            temperature=temperature,  # Temperature từ parameter (mặc định 0.9)
            top_p=0.92,  # Nucleus sampling - chỉ xét top 92% tokens
            top_k=50,  # Top-k sampling - chỉ xét top 50 tokens
            length_penalty=2.0,  # Khuyến khích generate dài
            repetition_penalty=1.2,  # Giảm lặp lại
            no_repeat_ngram_size=2,  # Tránh lặp cụm từ
            eos_token_id=test_tokenizer.eos_token_id,
            pad_token_id=test_tokenizer.pad_token_id
        )
    
    summary_raw = test_tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Debug: Tính toán thông tin về summary gốc
    original_summary_length = len(summary_raw.split())
    # Tính số tokens đã generate (loại bỏ input tokens)
    input_length = inputs['input_ids'].shape[1]
    output_length = output[0].shape[0]
    num_generated_tokens = output_length - input_length
    num_original_sentences_in_summary = len(re.findall(r'[.!?]', summary_raw))
    
    summary = summary_raw
    
    # Post-processing: Cắt ở câu hoàn chỉnh và loại bỏ phần không liên quan
    sentences = re.split(r'([.!?]+\s*)', summary)
    
    complete_sentences = []
    for i in range(0, len(sentences) - 1, 2):
        if i + 1 < len(sentences):
            sentence = (sentences[i] + sentences[i + 1]).strip()
            if sentence:
                complete_sentences.append(sentence)
    
    # Xử lý câu cuối - CHỈ giữ nếu có dấu kết thúc câu (để tránh cắt giữa câu)
    if len(sentences) % 2 == 1 and sentences[-1].strip():
        last_sentence = sentences[-1].strip()
        # CHỈ giữ nếu có dấu kết thúc câu (hoàn chỉnh)
        if re.search(r'[.!?]$', last_sentence):
            complete_sentences.append(last_sentence)
        # Nếu không có dấu kết thúc, bỏ qua (câu bị cắt)
    
    # Loại bỏ câu không liên quan hoặc lặp lại
    filtered_sentences = []
    seen_content = set()
    min_relevance_threshold = 0.05  # Giảm xuống để giữ nhiều câu hơn
    
    for sentence in complete_sentences:
        sentence_relevance = calculate_relevance_score(sentence, text)
        
        # Chỉ lọc câu có độ liên quan RẤT thấp (< 0.03) - rất lỏng để giữ nhiều nội dung
        if sentence_relevance < 0.03:
            continue
        
        normalized = re.sub(r'\s+', ' ', sentence.lower().strip())
        
        # Kiểm tra duplicate (tăng ngưỡng lên 0.9 để linh hoạt hơn, chỉ lọc câu gần như giống hệt)
        is_duplicate = False
        for seen in seen_content:
            if len(normalized) > 0 and len(seen) > 0:
                similarity = len(set(normalized.split()) & set(seen.split())) / max(len(normalized.split()), len(seen.split()))
                if similarity > 0.9:  # Tăng lên 0.9 để chỉ lọc câu gần như giống hệt
                    is_duplicate = True
                    break
        
        if not is_duplicate:
            filtered_sentences.append(sentence)
            seen_content.add(normalized)
    
    # Xử lý kết quả cuối cùng - GIỮ TẤT CẢ các câu hợp lệ, không dừng sớm
    if filtered_sentences:
        # Chỉ loại bỏ câu có độ liên quan RẤT thấp ở cuối (nếu có nhiều câu)
        # Nhưng vẫn giữ tất cả các câu hợp lệ
        final_summary = []
        for i, sentence in enumerate(filtered_sentences):
            sentence_relevance = calculate_relevance_score(sentence, text)
            
            # Chỉ bỏ qua câu có độ liên quan RẤT thấp (< 0.03) và chỉ khi đã có ít nhất 5 câu
            # Điều này đảm bảo giữ được nhiều nội dung
            if sentence_relevance < 0.03 and len(final_summary) >= 5:
                # Kiểm tra xem câu tiếp theo có tốt hơn không
                if i + 1 < len(filtered_sentences):
                    next_relevance = calculate_relevance_score(filtered_sentences[i+1], text)
                    if next_relevance < 0.03:
                        # Nếu câu tiếp theo cũng rất thấp, dừng ở đây
                        break
                else:
                    # Nếu đây là câu cuối, vẫn giữ nếu đã có ít nhất 5 câu
                    break
            
            final_summary.append(sentence)
        
        summary = ' '.join(final_summary)
    else:
        # Fallback: lấy các câu hoàn chỉnh từ summary gốc (có dấu kết thúc)
        sentences_original = re.split(r'([.!?]+\s*)', summary)
        fallback_sentences = []
        for i in range(0, len(sentences_original) - 1, 2):
            if i + 1 < len(sentences_original):
                sentence = (sentences_original[i] + sentences_original[i + 1]).strip()
                if sentence:
                    fallback_sentences.append(sentence)
        
        if fallback_sentences:
            summary = ' '.join(fallback_sentences)
        else:
            summary = summary.strip()
    
    # Debug output
    if debug:
        print(f"\n[DEBUG] Số câu gốc: {num_original_sentences}")
        print(f"[DEBUG] max_new_tokens: {max_new_tokens}, min_new_tokens: {min_new_tokens}")
        print(f"[DEBUG] Số tokens đã generate: {num_generated_tokens}")
        print(f"[DEBUG] Số từ trong summary gốc: {original_summary_length}")
        print(f"[DEBUG] Số câu trong summary gốc: {num_original_sentences_in_summary}")
        print(f"[DEBUG] Số câu sau post-processing: {len(final_summary) if filtered_sentences else len(fallback_sentences) if fallback_sentences else 0}")
    
    return summary

# ============================================
# TEST VỚI CÁC VÍ DỤ
# ============================================

print("\n" + "=" * 50)
print("TEST MODEL")
print("=" * 50)
test_text_3 = """TÌNH HÌNH XÃ HỘI GIAO CHÂU
Về mặt xã hội, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét cúa bọn quan lại đô hộ đã nhanh chóng đẩy người dân Giao Châu vào con đường bần cùng hóa. 
Sự phân hóa giai cấp trong xã hội ngày càng sâu sắc. Đặc biệt, từ nửa sau thế kỳ VIII, lụt lội và hạn hán xảy ra thường xuyên, chiến tranh liên tiếp do các nước láng giềng là Hoàn 
Vương (Champa) và Nam Chiếu gây ra, khiến sức sản xuất bị phá hoại, đời sống nhân dân ngày càng thêm cơ cực. về mặt văn hoá, nhà Đường đă đu nhập đạo Nho, đạo Phật và đạo Giáo với 
mục đích dựa vào sự phát triển văn hóa để nô dịch nhân dân Giao Châu. Nho giáo thời Đường ở Giao Châu chưa thể xem là phát triển, song cũng được truyền bá sâu rộng trong tầng lớp 
trên của xã hội. Trường học dạy chữ Hán được mờ nhiều ờ các phủ, châu. Trong tầng lớp Hào trưởng người Việt, một số gia đình đã cho con em học hành. Họ được tham gia thi cử và đỗ 
đạt ở Bắc triều, một số người được tuyển dụng vào bộ máy của chính quyền đô hộ, như trường hợp anh em Khương Công Phụ và Khương Công Phục. Tuy nhiên, việc học và thi cử ở Giao Châu 
vẫn bị hạn chế. Lệ nhà Đường năm Hội Xương thứ 5 (845) quy định: "An Nam đưa vào thi Tiến sĩ không được quá tám người, Minh kinh không được quá mười người". Dưới triều Đường, đạo 
Phật được truyền bá vào Giao Châu với hai phái Thiền tông của Phật giáo Trung Quốc. Phái thứ nhất do Tì Ni Đa Lưu Chi sáng lập, truyền bá vào Giao Châu cuối thế kỷ VI, trung tâm là 
chùa Pháp Vân (Thuận Thành, Bắc Ninh). Phái thứ hai do Vô Ngôn Thông sáng lập, truyền vào Giao Châu đầu thế kỷ IX, trung tâm là chùa Kiến Sơ (Phù Đổng, ngoại thành Hà Nội). Bên 
cạnh đạo Phật, Đạo giáo cũng khá phát triển và chịu ảnh hưởng sâu sắc của Trung Quốc. Nhà Đường đã cho nhiều đạo sĩ, phù thủy sang Giao Châu, trong đó có Tiết độ sứ Cao Biền với 
những phương thức tà ma, bùa chú để yểm trừ "long mạch"... Tuy nhiên, sự phát triển của đạo Nho, đạo Phật và Đạo giáo đều dung hòa được với tín ngưỡng dân gian cổ truyền của người 
Việt (như tục thờ thần sông, thần núi, thờ các vị anh hùng dân tộc.. tạo nên sức mạnh của dân tộc Việt, chống lại sự nô dịch văn hoá của ngoại bang."""
test_grade_3 = 1

print(f"\n" + "-" * 50)
print(f"[TEST 3] Lớp {test_grade_3}")
print(f"Văn bản gốc: {test_text_3}")
print(f"\nTóm tắt:")
summary_3 = summarize(test_text_3, test_grade_3, debug=True)
print(summary_3)

print("\n" + "=" * 50)
print("HOÀN TẤT TEST!")
print("=" * 50)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Đang load model từ: ../Model_DG/vit5_grade_summary
Đang load tokenizer từ checkpoint...
✓ Tokenizer loaded từ checkpoint (slow tokenizer)

Đang load model weights từ checkpoint...
✓ Model weights loaded từ checkpoint
Model đã được load vào: cuda

TEST MODEL

--------------------------------------------------
[TEST 3] Lớp 1
Văn bản gốc: TÌNH HÌNH XÃ HỘI GIAO CHÂU
Về mặt xã hội, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét cúa bọn quan lại đô hộ đã nhanh chóng đẩy người dân Giao Châu vào con đường bần cùng hóa. 
Sự phân hóa giai cấp trong xã hội ngày càng sâu sắc. Đặc biệt, từ nửa sau thế kỳ VIII, lụt lội và hạn hán xảy ra thường xuyên, chiến tranh liên tiếp do các nước láng giềng là Hoàn 
Vương (Champa) và Nam Chiếu gây ra, khiến sức sản xuất bị phá hoại, đời sống nhân dân ngày càng thêm cơ cực. về mặt văn hoá, nhà Đường đă đu nhập đạo Nho, đạo Phật và đạo Giáo với 
mục đích dựa vào sự phát triển văn hóa để nô dịch nhân dân Giao Châu. Nho giáo thời Đường ở G

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `num_beams` is set to 1. However, `length_penalty` is set to `2.0` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `length_penalty`.
  warnings.warn(



[DEBUG] Số câu gốc: 17
[DEBUG] max_new_tokens: 120, min_new_tokens: 60
[DEBUG] Số tokens đã generate: -439
[DEBUG] Số từ trong summary gốc: 66
[DEBUG] Số câu trong summary gốc: 3
[DEBUG] Số câu sau post-processing: 2
Ở Giao Châu, chính sách bóc lột dân sự và bóc tách giai cấp trong xã hội. Nhà Đường cho học hành nghiêm túc và thi cử tại Bắc triều.

HOÀN TẤT TEST!
